In [0]:
-- Determine ultimate beneficiaries for the Banking Suply Chain reporting

-- 01-1. Create a point in time run_timestamp for all data inserts
DROP TEMPORARY VARIABLE IF EXISTS run_timestamp;

DECLARE run_timestamp TIMESTAMP DEFAULT current_timestamp();

SELECT run_timestamp;


-- 01-2. Create default start date for all data inserts

DROP TEMPORARY VARIABLE IF EXISTS default_start_date;

DECLARE default_start_date STRING;

SET  VAR default_start_date = '01-01-1900';

SELECT default_start_date;


-- 01-3. Create default end date for all data inserts
DROP TEMPORARY VARIABLE IF EXISTS default_end_date;

DECLARE default_end_date STRING;

SET  VAR default_end_date = '31-12-2999';

SELECT default_end_date;

 
-- Determine ultimate beneficiaries
CREATE OR REPLACE TEMP VIEW view_third_level_beneficiary AS
SELECT tennant_id, originator_account_name, beneficiary_account_name, SUM(total_value) as total_value from 
demo_banking_gold.qdp_banking_supply_chain.view_banking_supply_chains
WHERE beneficiary_account_name NOT IN (
  SELECT originator_account_name
  FROM demo_banking_gold.qdp_banking_supply_chain.view_banking_supply_chains
)
GROUP BY tennant_id,originator_account_name, beneficiary_account_name
;

SELECT * FROM view_third_level_beneficiary;

-- Determine beneficiaries who are originators for ultimate beneficiaries
CREATE OR REPLACE TEMP VIEW view_second_level_beneficiary AS
SELECT tennant_id, originator_account_name, beneficiary_account_name, SUM(total_value) as total_value from 
demo_banking_gold.qdp_banking_supply_chain.view_banking_supply_chains
WHERE beneficiary_account_name IN (
  SELECT originator_account_name
  FROM view_third_level_beneficiary
)
GROUP BY tennant_id, originator_account_name, beneficiary_account_name
;

SELECT * FROM view_second_level_beneficiary;


DELETE FROM demo_banking_gold.qdp_banking_supply_chain.fact_cash_flow_3;

INSERT INTO demo_banking_gold.qdp_banking_supply_chain.fact_cash_flow_3 (tennant_id, stage1, stage2, stage3, value)
SELECT tennant_id, NULL, originator_account_name, beneficiary_account_name, total_value
 FROM view_third_level_beneficiary
;


INSERT INTO demo_banking_gold.qdp_banking_supply_chain.fact_cash_flow_3 (tennant_id, stage1, stage2, stage3, value)
SELECT tennant_id, originator_account_name, beneficiary_account_name, NULL, total_value
 FROM view_second_level_beneficiary
;


SELECT * FROM demo_banking_gold.qdp_banking_supply_chain.fact_cash_flow_3;


